# Vis-Viva Equation

This notebook contains the programmatic verification for the **Vis-Viva Equation** entry from the THEORIA dataset.

**Entry ID:** vis_viva  
**Required Library:** sympy 1.13.1

## Description
The Vis-Viva equation relates the velocity of a particle in an elliptical, hyperbolic or parabolic orbit to the distance to the barycenter, mass of the body it's orbiting, and the semi-major axis of the orbit. This sets the basis for velocity calculations in non-circular orbits and allows us to later derive equations like the equation for escape velocity with ease.

## Installation
First, let's install the required library:

In [ ]:
# Install required library with exact version
!pip install sympy==1.13.1

## Programmatic Verification

The following code verifies the derivation mathematically:

In [ ]:
import sympy as sp

# Step 1: Define specific energies at apoapsis and periapsis
G, M, a, r, v = sp.symbols('G M a r v', positive=True)
v_a, v_p, r_a, r_p = sp.symbols('v_a v_p r_a r_p', positive=True)
E_a = sp.Rational(1, 2) * v_a**2 - (G*M)/r_a
E_p = sp.Rational(1, 2) * v_p**2 - (G*M)/r_p
eq_energy = sp.Eq(E_a, E_p)

# Step 2: Rearranged energy equation
step2_expected = sp.Eq(sp.Rational(1, 2) * (v_a**2 - v_p**2), G*M*(1/r_a - 1/r_p))
assert sp.simplify(eq_energy.lhs - eq_energy.rhs - (step2_expected.lhs - step2_expected.rhs)) == 0

# Step 3: Angular momentum conservation gives v_p in terms of v_a, r_a, r_p
eq_angmom = sp.Eq(v_a*r_a, v_p*r_p)
v_p_from_L = sp.solve(eq_angmom, v_p)[0]
assert sp.simplify(v_p_from_L - (r_a*v_a)/r_p) == 0

# Step 4: Substitute step-3 expression into step-2 equation
eq_step4 = sp.simplify(step2_expected.subs(v_p, v_p_from_L))
step4_expected = sp.Eq(sp.Rational(1, 2) * (1 - (r_a/r_p)**2) * v_a**2, G*M*(1/r_a - 1/r_p))
residual_step4 = sp.simplify(eq_step4.lhs - eq_step4.rhs)
residual_expected = sp.simplify(step4_expected.lhs - step4_expected.rhs)
assert (sp.simplify(residual_step4 - residual_expected) == 0) or (sp.simplify(residual_step4 + residual_expected) == 0)
# Step 5: Solve for v_a^2/2
va2_over_2 = sp.simplify(sp.solve(eq_step4, v_a**2)[0] / 2)
step5_expected = sp.simplify(G*M * r_p / (r_a * (r_p + r_a)))
assert sp.simplify(va2_over_2 - step5_expected) == 0

# Step 6: Use r_p + r_a = 2a
va2_over_2_step6 = sp.simplify(va2_over_2.subs(r_p, 2*a - r_a))
step6_expected = sp.simplify(G*M/r_a - G*M/(2*a))
assert sp.simplify(va2_over_2_step6 - step6_expected) == 0

# Step 7: Orbital specific energy constant equals -GM/(2a)
E_constant = sp.simplify(E_a.subs(v_a**2, 2*va2_over_2_step6))
assert sp.simplify(E_constant + G*M/(2*a)) == 0

# Step 8: Recover vis-viva relation at arbitrary r
E_general = sp.Rational(1, 2) * v**2 - (G*M)/r
v2_expr = sp.simplify(sp.solve(sp.Eq(E_general, E_constant), v**2)[0])
vis_viva_expected = G*M*(2/r - 1/a)
assert sp.simplify(v2_expr - vis_viva_expected) == 0

print('Vis-viva programmatic verification passed')


## Source

📖 **View this entry:** [theoria-dataset.org/entries.html?entry=vis_viva.json](https://theoria-dataset.org/entries.html?entry=vis_viva.json)

This verification code is part of the [THEORIA dataset](https://github.com/theoria-dataset/theoria-dataset), a curated collection of theoretical physics derivations with programmatic verification.

**License:** CC-BY 4.0